# 🧠 Topic 5: Conversation Memory

## What I'm going to show you

In Topics 3 and 4 we built chatbot functions and refined our prompts. But there's a fundamental problem we've been ignoring: every API call we make is completely independent. The model has no idea what we said to it five seconds ago.

I'm going to show you exactly how that fails in a real Barclays scenario. Then I'll show you three things:

1. How to give the model memory by managing a messages list yourself
2. How to count tokens accurately with `tiktoken` so you can stay within budget
3. How to truncate the conversation safely when it gets too long

By the end, you'll have a `BarclaysChat` class that manages history internally and a `truncate_history()` function. Together they form the memory layer of the Barclays Product Knowledge Assistant.

## Learning Objectives

- Explain why the OpenAI API is stateless and why that matters
- Build a multi-turn chatbot by manually maintaining a messages list
- Count tokens accurately using `tiktoken` for `gpt-4o`
- Implement sliding-window truncation that preserves the system prompt
- Explain the tradeoff between truncation and summarization

## What component of the assistant are we building?

The **conversation memory layer**. This is the part that lets a customer ask "What is the interest rate on a Barclaycard?" and then follow up with "How do I apply for that?" without the assistant asking "apply for what?"


## Section 0: Environment Setup

Let's install everything we need. The key addition this topic is `tiktoken`, OpenAI's official tokenizer library. It lets me count tokens locally before making an API call, so I can make smart decisions about what to keep in the conversation history.


In [ ]:
# tiktoken: OpenAI's official tokenizer - counts tokens exactly as the API does.
# tiktoken==0.9.0 is pinned for classroom stability (0.12.0 dropped manylinux2014 wheels
# which can break older AL2-based SageMaker images).
# sagemaker==2.232.1 is the last classic SDK release where get_execution_role lives at the top level
# boto3 is the AWS SDK - needed for S3 and SageMaker session
# numpy<2 is the course-wide pin required for SageMaker compatibility.
!pip install -q "openai>=1.30.0" "tiktoken==0.9.0" "sagemaker==2.232.1" "boto3" "numpy<2"

In [ ]:
import os
import json
import getpass

import tiktoken
from openai import OpenAI

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

client = OpenAI()
MODEL = "gpt-4o"

# Reusing the T3 system prompt verbatim so the persona stays stable across topics.
BARCLAYS_SYSTEM_PROMPT = """You are a Barclays Product Knowledge Assistant.
Your role is to help customers understand Barclays products and services.

Persona and tone:
You are formal but warm - professional, clear, and friendly without being chatty.
You address the customer directly and use plain English. No jargon unless you
explain it in the same sentence.

Constraints - what you must not do:
- Only discuss Barclays products and services. If asked about competitors or
  unrelated topics, politely redirect: "I can only help with Barclays products
  and services."
- Provide product information only. You do not give personalised financial advice.
  If a customer asks which product is best for their situation, say:
  "I can share product details to help you compare, but for personalised advice
  please speak to a Barclays adviser."
- Never invent figures, rates, or terms. If the product information does not
  contain an answer, say: "I don't have that information to hand - please contact
  Barclays directly or visit barclays.co.uk for the latest details."

Format:
Keep answers to 3 sentences or fewer for simple factual questions.
For multi-part questions, answer each part in a short paragraph.
Do not use bullet lists unless the question explicitly asks for a list.
"""

print("Setup complete.")
print(f"Model: {MODEL}")
print(f"tiktoken version: {tiktoken.__version__}")


## What are we building today?

Here is the problem I want you to feel before I solve it.

A customer service agent is using our Barclays assistant and asks two questions in a row:

**Turn 1**: "What is the representative APR on the Barclaycard Avios Plus card?"

**Turn 2**: "How do I apply for that?"

If the assistant has no memory, turn 2 fails completely. "Apply for what?" The model has no idea what "that" refers to. From its perspective, turn 2 is the first message it has ever seen.

By the end of this topic you'll have a conversation loop that handles this naturally.

| | Topic 3 single-turn chatbot | Topic 5 multi-turn chatbot |
|---|---|---|
| Each call | Stateless, context is lost | Full history passed every turn |
| Turn 2 referent | "Apply for what?" | Correctly references turn 1 |
| Token cost | Constant per turn | Grows with conversation length |
| Memory strategy | None | Sliding-window truncation or summarization |


## Section 1: The Stateless API Problem

### Here is what breaks without memory

The OpenAI API does not remember anything between calls. Each `client.chat.completions.create()` call is completely independent. The model sees only what you send in that single request.

Let me show you what that looks like in practice with a real Barclays scenario.

```mermaid
graph TB
    subgraph BROKEN["Stateless: each call is independent"]
        direction TB
        U1A["User turn 1:<br/>What is the APR on the Barclaycard Avios Plus?"] --> C1A["API call 1<br/>messages = [system, user_1]"]
        C1A --> R1A["Assistant:<br/>Representative APR is 27.9 percent"]
        R1A --> U2A["User turn 2:<br/>How do I apply for that?"]
        U2A --> C2A["API call 2<br/>messages = [system, user_2]<br/><b>NO history of turn 1</b>"]
        C2A --> R2A["Assistant:<br/>Apply for what? I have no context."]
        R2A --> X1["FAIL: model cannot resolve 'that'"]
    end

    subgraph WORKING["Stateful: full history passed every call"]
        direction TB
        U1B["User turn 1:<br/>What is the APR on the Barclaycard Avios Plus?"] --> C1B["API call 1<br/>messages = [system, user_1]"]
        C1B --> R1B["Assistant:<br/>Representative APR is 27.9 percent"]
        R1B --> U2B["User turn 2:<br/>How do I apply for that?"]
        U2B --> C2B["API call 2<br/>messages = [system, user_1,<br/>assistant_1, user_2]<br/><b>Full history included</b>"]
        C2B --> R2B["Assistant:<br/>To apply for the Avios Plus,<br/>visit barclaycard.co.uk/apply"]
        R2B --> OK1["SUCCESS: model resolves 'that' = Avios Plus"]
    end

    style X1 fill:#ffcccc,stroke:#cc0000,stroke-width:2px
    style OK1 fill:#ccffcc,stroke:#006600,stroke-width:2px
    style C1A fill:#fff5cc
    style C2A fill:#fff5cc
    style C1B fill:#cce5ff
    style C2B fill:#cce5ff
```

The diagram above shows two isolated calls. The model answers turn 1 fine. But when turn 2 arrives asking "how do I apply for that?", the model has no idea what "that" refers to. It never saw turn 1.


In [ ]:
# BROKEN: Two separate calls with no history.
# This is what happens if we call the API twice without passing context.
# The second call has no idea what the first was about.

print("=== Turn 1 ===")
response_1 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": BARCLAYS_SYSTEM_PROMPT},
        {"role": "user",   "content": "What is the representative APR on the Barclaycard Avios Plus card?"}
    ]
)
answer_1 = response_1.choices[0].message.content
print(f"Assistant: {answer_1}\n")

print("=== Turn 2 (new call, no history) ===")
response_2 = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": BARCLAYS_SYSTEM_PROMPT},
        {"role": "user",   "content": "How do I apply for that?"}
    ]
)
answer_2 = response_2.choices[0].message.content
print(f"Assistant: {answer_2}")
print("\n*** Notice: the model has no idea what 'that' refers to. ***")
print("*** This is the stateless problem. Each call starts from scratch. ***")


### The fix: pass the full conversation history every time

The solution is simple but important. After each turn, I append both the user message and the assistant response to a list. Then I pass the entire list with every new API call.

```mermaid
graph TD
    Start["Initial state<br/>history = [system_msg]<br/>length = 1"] --> T1U["Turn 1: user says<br/>'What is the APR on Avios Plus?'"]

    T1U --> T1Append1["history.append(user_1)<br/>length = 2"]
    T1Append1 --> T1Send["Send full history to API<br/>[system, user_1]"]
    T1Send --> T1Reply["Assistant reply:<br/>'27.9 percent representative'"]
    T1Reply --> T1Append2["history.append(assistant_1)<br/>length = 3"]

    T1Append2 --> T2U["Turn 2: user says<br/>'How do I apply for that?'"]
    T2U --> T2Append1["history.append(user_2)<br/>length = 4"]
    T2Append1 --> T2Send["Send full history to API<br/>[system, user_1, assistant_1, user_2]<br/><b>Model sees turn 1 = can resolve 'that'</b>"]
    T2Send --> T2Reply["Assistant reply:<br/>'Apply at barclaycard.co.uk/apply'"]
    T2Reply --> T2Append2["history.append(assistant_2)<br/>length = 5"]

    T2Append2 --> T3U["Turn 3: user says<br/>'What is the annual fee?'"]
    T3U --> T3Append1["history.append(user_3)<br/>length = 6"]
    T3Append1 --> T3Send["Send full history to API<br/>[system, u1, a1, u2, a2, u3]<br/><b>Full context preserved</b>"]
    T3Send --> T3Reply["Assistant reply:<br/>'240 GBP per year for the Avios Plus'"]
    T3Reply --> Done["After turn 3:<br/>history length = 7<br/>(grows by 2 every turn)"]

    style Start fill:#e0e0e0
    style T1Send fill:#cce5ff
    style T2Send fill:#cce5ff
    style T3Send fill:#cce5ff
    style Done fill:#fff5cc,stroke:#cc8800,stroke-width:2px
```

The messages list grows with every turn. Turn 1 adds 2 messages (user + assistant). Turn 2 adds 2 more. By turn 5 the list has 10 messages plus the system prompt. The model sees the entire conversation and can answer referential questions correctly.


In [ ]:
# WORKING: Full conversation history passed every call.
# The key change: I maintain a 'history' list and append every turn to it.

def chat_with_history_demo(user_message, history):
    """
    Send a user message with the full conversation history.
    Returns the assistant reply and the updated history list.

    Parameters
    ----------
    user_message : str
        The new user turn to add.
    history : list
        The current conversation history (list of role/content dicts).
        Must start with the system message if this is the first call.

    Returns
    -------
    reply : str
        The assistant's response text.
    history : list
        The updated history including this turn and the assistant reply.
    """
    # Step 1: append the user message to history.
    history.append({"role": "user", "content": user_message})

    # Step 2: call the API with the FULL history, not just the latest message.
    response = client.chat.completions.create(
        model=MODEL,
        messages=history
    )

    # Step 3: extract the reply text.
    reply = response.choices[0].message.content

    # Step 4: append the assistant's reply to history so future turns see it.
    # This is the step beginners most often forget.
    history.append({"role": "assistant", "content": reply})

    return reply, history


history = [{"role": "system", "content": BARCLAYS_SYSTEM_PROMPT}]

print("=== Turn 1 ===")
reply, history = chat_with_history_demo(
    "What is the representative APR on the Barclaycard Avios Plus card?",
    history
)
print(f"Assistant: {reply}\n")

print("=== Turn 2 (full history passed) ===")
reply, history = chat_with_history_demo(
    "How do I apply for that?",
    history
)
print(f"Assistant: {reply}")
print(f"\nHistory now has {len(history)} messages (1 system + 2 pairs of user+assistant).")


### Lab 1: Build the BarclaysChat class

Now it's your turn. I want you to build a cleaner version of the function above, wrapped in a class that keeps the history internally so callers do not have to pass it around themselves.

**What to build**: A `BarclaysChat` class with:

- `__init__(self, system_prompt)`: initialises `self.history` with the system message
- `chat(self, user_message)`: appends user message, calls API, appends reply, returns reply text
- `reset(self)`: clears history back to just the system prompt

**Verification**: Run the 3-turn Barclays scenario at the bottom of the starter cell. Turn 3 asks about the annual fee for a specific card. A correct answer requires memory of turns 1 and 2.

**Stretch (fast finishers)**: Add a `show_history()` method that prints each turn formatted nicely. Role in uppercase, content truncated to 100 chars if long.

**Homework Extension**: Extend `BarclaysChat` to persist `self.history` to a JSON file after every turn, and load it on init if the file exists. This simulates a session that survives a kernel restart, critical for any production chatbot.


In [ ]:
# Lab 1: Build the BarclaysChat class
# Implement __init__, chat, and reset. The verification at the bottom
# runs a 3-turn conversation that requires memory to make sense.

class BarclaysChat:
    """
    A stateful chatbot that maintains full conversation history internally.
    Implement the three methods below.
    """

    def __init__(self, system_prompt):
        # Step 1: store the system prompt on the instance so reset() can rebuild from it
        # Step 2: initialise self.history as a list containing one dict:
        #         {"role": "system", "content": system_prompt}
        self.history = None  # YOUR CODE

    def chat(self, user_message):
        """Send a message and return the assistant reply."""
        # Step 1: append a user message dict to self.history
        # Step 2: call client.chat.completions.create with model=MODEL and messages=self.history
        # Step 3: extract reply from response.choices[0].message.content
        # Step 4: append the assistant reply dict to self.history
        # Step 5: return the reply
        reply = None  # YOUR CODE
        return reply

    def reset(self):
        """Clear history back to just the system prompt."""
        # Re-initialise self.history with only the system message
        pass  # YOUR CODE


# Verification
# Run this after implementing the class above.
# All three turns should make sense as a coherent conversation.

bot = BarclaysChat(BARCLAYS_SYSTEM_PROMPT)

turn1 = bot.chat("What Barclaycard products do you offer for someone who travels frequently?")
print(f"Turn 1: {turn1[:200] if turn1 else '(no reply - implement chat() first)'}\n")

turn2 = bot.chat("Which of those earns the most Avios points per pound spent?")
print(f"Turn 2: {turn2[:200] if turn2 else '(no reply)'}\n")

turn3 = bot.chat("What is the annual fee for that card?")
print(f"Turn 3: {turn3[:200] if turn3 else '(no reply)'}\n")

# This should reference the specific card mentioned in turn 2, not say "which card?"
print(f"History length: {len(bot.history) if isinstance(bot.history, list) else 'not a list'} messages")


In [ ]:
# Safety-net: if you did not finish the lab above, run this cell so the rest of the
# notebook still works. If you did finish it, SKIP this cell.
# Fires when bot.history is not a populated list (i.e. __init__ was not implemented).

if not isinstance(getattr(bot, "history", None), list) or bot.history is None or len(bot.history) <= 1:
    print("Lab 1 not complete - using safety-net BarclaysChat for remaining cells.")

    class BarclaysChat:
        def __init__(self, system_prompt):
            self.system_prompt = system_prompt
            self.history = [{"role": "system", "content": system_prompt}]

        def chat(self, user_message):
            self.history.append({"role": "user", "content": user_message})
            response = client.chat.completions.create(
                model=MODEL, messages=self.history
            )
            reply = response.choices[0].message.content
            self.history.append({"role": "assistant", "content": reply})
            return reply

        def reset(self):
            self.history = [{"role": "system", "content": self.system_prompt}]

    bot = BarclaysChat(BARCLAYS_SYSTEM_PROMPT)
    bot.chat("What Barclaycard products do you offer for someone who travels frequently?")
    bot.chat("Which of those earns the most Avios points per pound spent?")
    bot.chat("What is the annual fee for that card?")
    print(f"Safety-net bot ready. History length: {len(bot.history)} messages.")
else:
    print(f"Lab 1 complete. Bot history: {len(bot.history)} messages. Moving on.")


## Section 2: Token Counting and Sliding-Window Truncation

### The history keeps growing, and that costs money

Every turn adds more messages to the history. `gpt-4o` charges you for every input token you send. A 50-turn customer service conversation can easily have 10,000+ input tokens in every single API call. That is just history overhead, before the user has even said anything new.

I need two things:

1. A way to count tokens accurately before making a call
2. A strategy to trim the history when it gets too long

Let me show you how to use `tiktoken` to count tokens for a messages list.

The formula for `gpt-4o` (which uses the `o200k_base` encoding):

- 3 tokens per message (role separator overhead)
- Tokens in the actual content string
- 3 priming tokens added at the end for the reply

```mermaid
graph TB
    Input["history (8 messages, ~3500 tokens)<br/>[system_msg, u1, a1, u2, a2, u3, a3, u4]<br/>BUDGET: 1500 tokens"]

    Input --> Naive
    Input --> Safe

    subgraph Naive["NAIVE: history[1:] - drops oldest including system"]
        direction TB
        N1["Step 1: drop history[0]<br/>history[1:] = [u1, a1, u2, a2, u3, a3, u4]<br/>~3300 tokens - still over budget"]
        N1 --> N2["Step 2: drop again<br/>[a1, u2, a2, u3, a3, u4]<br/>~2900 tokens - still over"]
        N2 --> N3["...keep dropping from the front..."]
        N3 --> N4["Final: [u3, a3, u4]<br/>~1400 tokens - fits<br/><b>SYSTEM PROMPT GONE</b>"]
        N4 --> NX["FAIL: assistant lost its persona<br/>No more 'Barclays product expert'<br/>Generic chatbot, no guardrails"]
    end

    subgraph Safe["SAFE: [system_msg] + drop oldest non-system"]
        direction TB
        S1["Step 1: peel off system_msg<br/>conversation = [u1, a1, u2, a2, u3, a3, u4]<br/>system kept aside"]
        S1 --> S2["Step 2: drop conversation[0]<br/>[a1, u2, a2, u3, a3, u4]<br/>~3000 tokens with system - still over"]
        S2 --> S3["...keep popping oldest non-system..."]
        S3 --> S4["Final: [system_msg] + [u3, a3, u4]<br/>~1450 tokens - fits<br/><b>SYSTEM PROMPT PRESERVED</b>"]
        S4 --> SOK["SUCCESS: assistant keeps its persona<br/>Still a Barclays product expert<br/>Guardrails intact, recent context kept"]
    end

    style NX fill:#ffcccc,stroke:#cc0000,stroke-width:3px
    style SOK fill:#ccffcc,stroke:#006600,stroke-width:3px
    style N4 fill:#ffe5e5
    style S4 fill:#e5ffe5
    style Input fill:#fff5cc,stroke:#cc8800,stroke-width:2px
```

The diagram shows the critical mistake: naive truncation with `messages[1:]` drops the system prompt. The system prompt defines the assistant's persona and constraints. Losing it turns our Barclays assistant into a generic chatbot with no guardrails.


In [ ]:
# Token counting with tiktoken.
# tiktoken gives us the exact same tokenizer the API uses, so our counts match.

def count_tokens_in_messages(messages, model="gpt-4o"):
    """
    Count the total tokens in a messages list, including per-message overhead.

    Formula for gpt-4o (o200k_base encoding):
      - 3 tokens per message (role + separator overhead)
      - tokens in each field value (role name, content string)
      - 3 priming tokens at the end (reply initialization)

    Note: counts are estimates. For exact billing figures use response.usage
    after a real API call.

    Parameters
    ----------
    messages : list of dict
        The messages list in OpenAI chat format.
    model : str
        Model name, used to select the correct encoding.

    Returns
    -------
    int
        Estimated total token count.
    """
    try:
        # Get the exact tokenizer for this model (gpt-4o uses o200k_base).
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        # Fall back to o200k_base if the model is not in tiktoken's registry.
        encoding = tiktoken.get_encoding("o200k_base")

    tokens_per_message = 3
    num_tokens = 0

    for message in messages:
        num_tokens += tokens_per_message
        for key, value in message.items():
            num_tokens += len(encoding.encode(value))

    num_tokens += 3
    return num_tokens


print(f"History after 3 turns: {len(bot.history)} messages")
token_count = count_tokens_in_messages(bot.history)
print(f"Estimated token count: {token_count} tokens")

print("\n--- Per-message breakdown ---")
enc = tiktoken.encoding_for_model("gpt-4o")
for i, msg in enumerate(bot.history):
    content_tokens = len(enc.encode(msg["content"]))
    role_tokens = len(enc.encode(msg["role"]))
    total = 3 + role_tokens + content_tokens
    label = msg["content"][:50].replace("\n", " ")
    print(f"  [{i}] {msg['role']:10s} | {total:4d} tokens | {label}...")


### Lab 2: Implement truncate_history()

Now build the safe truncation function. The function must:

1. Always keep the system message (index 0 of the history list)
2. Keep only the most recent messages that fit within a token budget
3. Drop oldest non-system messages first, one at a time, until the budget fits

**Important**: the naive mistake is `messages[-N:]` which can drop the system prompt. The correct pattern is `[system_msg] + recent_messages`.

**Verification**: The test at the bottom of the starter cell builds a 15-turn history then truncates it to 1500 tokens. The output should have fewer messages but the system prompt must still be present at index 0, and the total tokens must be at or below 1500.

**Stretch (fast finishers)**: Modify the function to only drop complete user/assistant pairs, never truncating in the middle of a pair. Hint: after the system message, messages come in pairs (user, assistant), so when you trim, trim 2 at a time from the oldest end.

**Homework Extension**: Add per-turn cost tracking to `BarclaysChat`. After each call, read `response.usage.prompt_tokens` and `response.usage.completion_tokens` and accumulate a running total. Add a `cost_report()` method that prints tokens used and estimated cost at gpt-4o rates ($2.50 per million input tokens, $10.00 per million output tokens). This is exactly the kind of cost observability production teams need.


In [ ]:
# Lab 2: Implement truncate_history
# Build a sliding-window truncation function that always preserves the
# system message and drops oldest non-system messages until the budget fits.

def truncate_history(history, max_tokens, model="gpt-4o"):
    """
    Trim a conversation history to fit within max_tokens.
    Always preserves the system message at index 0.
    Removes oldest non-system messages first.

    Parameters
    ----------
    history : list of dict
        Full conversation history. history[0] must be the system message.
    max_tokens : int
        Maximum token budget for the returned history.
    model : str
        Model name for tiktoken encoding selection.

    Returns
    -------
    list of dict
        Trimmed history that fits within max_tokens, always starting with
        the system message.
    """
    # Step 1: separate the system message from the rest
    # Hint: history[0] is the system message, history[1:] is the conversation
    system_msg = None        # YOUR CODE
    conversation = None      # YOUR CODE

    # Step 2: drop oldest conversation messages until the total fits the budget
    # Hint: while count_tokens_in_messages([system_msg] + conversation) > max_tokens
    #       and len(conversation) > 0: remove conversation[0]
    # YOUR CODE

    # Step 3: return [system_msg] + remaining conversation
    return None  # YOUR CODE


# Verification
# Build a long history with 15 turns to test truncation.

long_bot = BarclaysChat(BARCLAYS_SYSTEM_PROMPT)
questions = [
    "What Barclaycard options are available?",
    "Tell me about the Barclaycard Avios Plus.",
    "What is the representative APR?",
    "What is the credit limit range?",
    "Is there a balance transfer offer?",
    "What is the balance transfer fee?",
    "How do I apply?",
    "What documents do I need?",
    "How long does approval take?",
    "Can I manage it through the app?",
    "What is the annual fee?",
    "Are there any additional benefits?",
    "What is the foreign transaction fee?",
    "Is there a minimum income requirement?",
    "How do I cancel if I need to?"
]
for q in questions:
    long_bot.chat(q)

full_tokens = count_tokens_in_messages(long_bot.history)
print(f"Full history: {len(long_bot.history)} messages, ~{full_tokens} tokens")

MAX_TOKENS = 1500
trimmed = truncate_history(long_bot.history, max_tokens=MAX_TOKENS)

if trimmed is not None:
    trimmed_tokens = count_tokens_in_messages(trimmed)
    print(f"Trimmed history: {len(trimmed)} messages, ~{trimmed_tokens} tokens")
    print(f"System prompt preserved: {trimmed[0]['role'] == 'system'}")
    print(f"Fits within budget: {trimmed_tokens <= MAX_TOKENS}")
else:
    print("truncate_history() returned None - check your implementation.")


In [ ]:
# Safety-net: provides a working truncate_history for the summarization demo and
# the end-to-end test below. Fires if the lab implementation returned None or
# truncate_history is missing entirely.

def _reference_truncate_history(history, max_tokens, model="gpt-4o"):
    """Reference implementation used as fallback only."""
    system_msg = history[0]
    conversation = list(history[1:])
    while (count_tokens_in_messages([system_msg] + conversation) > max_tokens
           and len(conversation) > 0):
        conversation.pop(0)
    return [system_msg] + conversation

if "trimmed" not in dir() or trimmed is None:
    print("Lab 2 not complete - using safety-net truncate_history for remaining cells.")
    truncate_history = _reference_truncate_history
    trimmed = truncate_history(long_bot.history, max_tokens=1500)
    print(f"Safety-net trimmed history: {len(trimmed)} messages.")
else:
    print("Lab 2 complete. Moving on.")


## Section 3: Summarization-Based Compression

### When truncation is not good enough

Sliding-window truncation is fast and simple, but it throws away context permanently. If the user mentioned in turn 1 that they are a Barclaycard Gold customer, and we truncate past turn 10, that context is gone forever.

Summarization is the alternative. Instead of deleting old turns, I compress them into a short summary and inject that summary back into the history. The model still has access to the key facts from early in the conversation, but at a fraction of the token cost.

**The tradeoff:**

| | Truncation | Summarization |
|---|---|---|
| Speed | Fast (no extra API call) | Slower (requires one extra call) |
| Memory quality | Loses old context entirely | Preserves key facts in compressed form |
| Cost | Saves tokens immediately | Adds one API call, saves tokens long-term |
| Risk | Simple, predictable | Summary may lose specific details |

For a banking chatbot, summarization is preferred when early turns contain product specifics (account type, amounts discussed, decisions made) that remain relevant later. There is no lab for this section. I'm walking you through a working demo so you understand the pattern. Implementing it as production-grade code is a homework extension at the bottom of the notebook.


In [ ]:
# Summarization-based memory compression.
# When the history gets too long, we compress old turns into a summary and
# replace them with a single assistant message containing that summary.

def summarize_and_compress(history, keep_recent=4, model="gpt-4o"):
    """
    Compress old conversation turns into a summary, keeping the most recent turns verbatim.

    Strategy:
    1. Keep the system message and the most recent 'keep_recent' messages as-is.
    2. Send all older turns to gpt-4o and ask for a concise summary.
    3. Replace the old turns with a single assistant message containing the summary.

    Parameters
    ----------
    history : list of dict
        The full conversation history. history[0] is the system message.
    keep_recent : int
        Number of most recent messages to keep verbatim (not summarized).
    model : str
        Model to use for the summarization call.

    Returns
    -------
    list of dict
        Compressed history: system msg + summary msg + recent verbatim turns.
    """
    if len(history) <= keep_recent + 2:
        return history

    system_msg = history[0]
    old_turns  = history[1:-keep_recent]
    recent     = history[-keep_recent:]

    if not old_turns:
        return history

    transcript = "\n".join(
        f"{m['role'].upper()}: {m['content']}" for m in old_turns
    )

    # Ask gpt-4o to summarise, preserving the key facts a banking assistant needs.
    summarize_prompt = [
        {"role": "system", "content": (
            "You are a conversation summarizer for a banking chatbot. "
            "Compress the conversation below into a concise summary of at most 150 words. "
            "Preserve: product names, account types, specific figures (APR, fees, amounts), "
            "and any decisions or commitments made. "
            "Discard pleasantries and repetition."
        )},
        {"role": "user", "content": f"Conversation to summarize:\n\n{transcript}"}
    ]

    summary_response = client.chat.completions.create(
        model=model,
        messages=summarize_prompt,
        max_tokens=200
    )
    summary_text = summary_response.choices[0].message.content

    # Build the compressed history:
    # system prompt + summary as an assistant message + recent verbatim turns.
    compressed = [
        system_msg,
        {"role": "assistant",
         "content": f"[Conversation summary: {summary_text}]"}
    ] + list(recent)

    return compressed


# Demo: compress the 15-turn history we built in Lab 2.
print(f"Before compression: {len(long_bot.history)} messages")
print(f"Token count: ~{count_tokens_in_messages(long_bot.history)} tokens\n")

compressed_history = summarize_and_compress(long_bot.history, keep_recent=4)

print(f"After compression: {len(compressed_history)} messages")
print(f"Token count: ~{count_tokens_in_messages(compressed_history)} tokens\n")

print("--- Summary message injected ---")
print(compressed_history[1]["content"])


## Wrap-Up: What I built today

Today I added the memory layer to the Barclays Product Knowledge Assistant.

**What you now have:**

- `BarclaysChat` - a stateful chatbot class that appends every turn to a history list
- `count_tokens_in_messages()` - accurate token counting using tiktoken's `o200k_base` encoding
- `truncate_history()` - safe sliding-window truncation that always preserves the system prompt
- `summarize_and_compress()` - a demonstration of summarization-based compression

**The key insight:**

The OpenAI API is stateless. Memory is your responsibility as the application developer. The messages list is not a convenience, it is the entire memory mechanism.

**What comes next: Topic 6 - RAG Foundations**

Instead of relying on the model's training data to know about Barclays products, we will retrieve relevant product information from a vector store and inject it into the conversation at each turn. The `BarclaysChat` class you built today will be extended to accept retrieved context.

**Take-home extensions:**

1. Add per-turn cost tracking using `response.usage.prompt_tokens` and `response.usage.completion_tokens` (rates: $2.50 per 1M input tokens, $10.00 per 1M output tokens for gpt-4o).
2. Add session persistence: save `self.history` to a JSON file after each turn and reload it on init if the file exists.
3. Implement a hybrid strategy: keep verbatim for the last 20 turns, summarize everything older. This is the production-ready pattern used by most commercial chatbots.

**Production note:**

In a financial services context (like Barclays), conversation logs may contain customer names, account types, and product details. Never store raw conversation history without PII scrubbing and a defined retention window. UK GDPR and FCA Consumer Duty frameworks apply to conversation data. Always verify actual token costs from `response.usage`. tiktoken estimates are accurate for pure text but can diverge slightly when tool calls or function schemas are present.

**Prompt caching tip:**

OpenAI automatically caches stable prompt prefixes at a 50% token discount. Keeping your system prompt consistent and at the top of every call maximizes cache hit rate. Exactly the pattern we use in `BarclaysChat`.


In [ ]:
# End-to-end test: 5 turns through the full memory pipeline.
# This cell ties everything together: history management + token counting + truncation.

MAX_HISTORY_TOKENS = 800   # keep the history lean for this demo

test_bot = BarclaysChat(BARCLAYS_SYSTEM_PROMPT)

test_conversation = [
    "What is the Barclaycard Avios Plus card?",
    "What is the annual fee and representative APR?",
    "How do I apply for that card?",
    "What credit score do I need?",
    "How do I apply for that?"
]

print("=== Full memory pipeline demo ===\n")
for i, question in enumerate(test_conversation, 1):
    # Truncate before each call to keep history within budget.
    test_bot.history = truncate_history(test_bot.history, max_tokens=MAX_HISTORY_TOKENS)

    reply = test_bot.chat(question)
    token_count = count_tokens_in_messages(test_bot.history)

    print(f"Turn {i}: {question}")
    print(f"Reply:  {reply[:150]}...")
    print(f"History: {len(test_bot.history)} messages, ~{token_count} tokens\n")

print("The final turn ('How do I apply for that?') should reference the Barclaycard Avios Plus card.")
print("If it does, memory is working correctly across all 5 turns.")
